In [1]:
from bokeh import layouts
from bokeh import models
from bokeh.io import output_notebook, show
from bokeh.plotting import figure, column
from bokeh.models import Range1d, Div, BoxSelectTool

import pandas as pd
import numpy as np

output_notebook()


def performance_metrics(pnl) -> dict[str, float]:
    return dict(
        total_pnl = np.sum(pnl),
        average_pnl = np.mean(pnl),
        average_win = np.mean(pnl[pnl > 0]),
        average_loss = np.mean(pnl[pnl < 0]),
        count = np.size(pnl),
        win_rate = np.mean(pnl > 0),
        loss_rate = np.mean(pnl < 0),
        zero_rate = np.mean(pnl == 0),
        oracle_score = np.sum(pnl) / np.sum(np.abs(pnl)),
        sharpe_ratio = np.mean(pnl) / np.std(pnl),
        sortino_ratio = np.mean(pnl) / np.std(pnl[pnl < 0]),
        maximum_drawdown = np.max(np.maximum.accumulate(np.cumsum(pnl)) - np.cumsum(pnl)),
        maximum_drawdown_relative_to_total_pnl = np.max(np.maximum.accumulate(np.cumsum(pnl)) - np.cumsum(pnl)) / np.sum(pnl),
    )


def performance_metrics_div(pnl):
    return Div(
        text=(
            "<p>"
            + "<br>".join(
                f"<b>{name.replace('_', ' ').title()}</b>: {value:.2f}"
                for name, value in performance_metrics(pnl).items()
            )
            + "</p>"
        )
    )

Loading BokehJS ...

In [13]:
import pandas as pd
import numpy as np


alpha = 5
deadband=0.003
window = 60


sp500 = (
    pd.read_csv("sp500.csv")
    .rename(columns={"date": "timestamp", "raw": "price", "log": "log_price"})
    .assign(timestamp=lambda x: pd.to_datetime(x.timestamp))
)

backtest = (
    sp500
    .assign(
        expected_return=lambda x: ((x.price.shift().rolling(window).mean() - x.price.shift()) / x.price.shift()).fillna(0)
    )
    .assign(
        position=lambda x: np.where(np.abs(x.expected_return) <= deadband, 0, np.clip(alpha * x.expected_return, -1, 1))
    )
    .assign(
        spread=lambda x: x.price.diff().fillna(0),
    )
    .assign(
        pnl=lambda x: x.position * x.spread,
        base_long_pnl=lambda x: +1 * x.spread,
        base_short_pnl=lambda x: -1 * x.spread,
    )
    .assign( 
        cummulative_pnl=lambda x: np.cumsum(x.pnl),
        cummulative_base_long_pnl=lambda x: np.cumsum(x.base_long_pnl),
        cummulative_base_short_pnl=lambda x: np.cumsum(x.base_short_pnl),
    )
)


assert not np.isnan(backtest.expected_return).any()
assert not np.isnan(backtest.position).any()
assert not np.isnan(backtest.spread).any()
assert not np.isnan(backtest.pnl).any()


tools = "pan,box_select,wheel_zoom,box_zoom,reset"

source = models.ColumnDataSource(backtest)
pnl_plot = figure(
    x_axis_type="datetime",
    width=1000,
    height=400,
    title="Cummulative PnL",
    background_fill_color="#efefef",
    y_axis_label="PnL",
    tools=tools,
)
pnl_plot.toolbar.logo = None
pnl_plot.add_tools(models.HoverTool(
    tooltips=[
        ("timestamp", "@timestamp{%Y-%m-%d %H:%M:%S}"),
        ("pnl", "@pnl"),
    ],
    formatters={"@timestamp": "datetime"}
))
pnl_plot.xaxis.axis_label_text_font_style = "normal"
pnl_plot.yaxis.axis_label_text_font_style = "normal"
pnl_plot.xaxis.major_label_orientation = np.pi / 4  # radians
pnl_plot.scatter("timestamp", "cummulative_pnl", source=source, selection_color="firebrick")
pnl_plot.scatter("timestamp", "cummulative_base_short_pnl", source=source, color="red")
pnl_plot.scatter("timestamp", "cummulative_base_long_pnl", source=source, color="green")


position_plot = figure(
    x_axis_type="datetime",
    width=1000,
    height=400,
    title="Position",
    background_fill_color="#efefef",
    y_axis_label="Position",
    tools=tools,
)
position_plot.toolbar.logo = None
position_plot.xaxis.axis_label_text_font_style = "normal"
position_plot.yaxis.axis_label_text_font_style = "normal"
position_plot.xaxis.major_label_orientation = np.pi / 4  # radians
position_plot.scatter("timestamp", "position", source=source, selection_color="firebrick")


sizer_plot = figure(
    width=1000,
    height=400,
    title="Sizer",
    background_fill_color="#efefef",
    x_axis_label="Expected Return",
    y_axis_label="Position",
    tools=tools,
)
sizer_plot.toolbar.logo = None
sizer_plot.xaxis.axis_label_text_font_style = "normal"
sizer_plot.yaxis.axis_label_text_font_style = "normal"
sizer_plot.xaxis.major_label_orientation = np.pi / 4  # radians
sizer_plot.scatter("expected_return", "position", source=source, selection_color="firebrick")

layout = column(
    performance_metrics_div(backtest.pnl),
    pnl_plot,
    position_plot,
    sizer_plot,
)

show(layout)

In [2]:
from bokeh.layouts import gridplot
from bokeh.models import ColumnDataSource
from bokeh.plotting import figure, show
from bokeh.sampledata.penguins import data
from bokeh.transform import factor_cmap

SPECIES = sorted(data.species.unique())

TOOLS = "box_select,lasso_select,help"

source = ColumnDataSource(data)

left = figure(width=300, height=400, title=None, tools=TOOLS,
              background_fill_color="#fafafa")
left.scatter("bill_length_mm", "body_mass_g", source=source,
             color=factor_cmap('species', 'Category10_3', SPECIES))

right = figure(width=300, height=400, title=None, tools=TOOLS,
               background_fill_color="#fafafa", y_axis_location="right")
right.scatter("bill_depth_mm", "body_mass_g", source=source,
              color=factor_cmap('species', 'Category10_3', SPECIES))

show(gridplot([[left, right]]))